# Convolutional Neural Network: 
## Food Recognition

In [60]:
import numpy as np
from pathlib import Path
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()

from copy import deepcopy

# Progress bar
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torchvision import datasets, transforms
from torchvision import transforms

import tensorboard as tb
from torch.utils.tensorboard import SummaryWriter

In [61]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## CNN Structure:

In [ ]:
class CNN(nn.Module):

    def __init__(self):
        """ Initializing the MLP module """
        super().__init__()
        self.net = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(128 * 28 * 28, 256),  # depends on input size
            nn.ReLU(),

            nn.Linear(256, 80)
        )

    def forward(self, x):
        out = self.net(x)
        return out

    @property
    def device(self):
        """
        Returns the device on which the model is. Can be useful in some situations.
        """
        return next(self.parameters()).device

### Seed

In [63]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): # GPU operation have separate seed
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# Additionally, some operations on a GPU are implemented stochastic for efficiency
# We want to ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

Using device cpu


### Evaluation

In [64]:
def eval_model(model, data_loader):

    model.eval()

    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)

            preds = outputs.argmax(dim=1)

            total_correct += (preds == targets).sum().item()
            total_samples += targets.size(0)

    avg_accuracy = total_correct / total_samples

    return avg_accuracy

### Analyze Data

In [ ]:
# Paths
ROOT = Path("food-recognition-challenge-2026")

TRAIN_DIR = ROOT / "train_set"
TEST_DIR  = ROOT / "test_set"
TRAIN_CSV = ROOT / "train_labels.csv"
CLASS_TXT = ROOT / "class_list_food.txt"

# aantal files
train_files = list(TRAIN_DIR.rglob("*"))
test_files  = list(TEST_DIR.rglob("*"))

print("Train total items:", len(train_files))
print("Test total items:", len(test_files))

# alleen images (op extensie)
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
train_imgs = [p for p in TRAIN_DIR.rglob("*") if p.suffix.lower() in img_exts]
test_imgs  = [p for p in TEST_DIR.rglob("*") if p.suffix.lower() in img_exts]

print("Train images:", len(train_imgs))
print("Test images:", len(test_imgs))

from collections import Counter
print("Train extensions:", Counter([p.suffix.lower() for p in train_imgs]).most_common())
print("Test extensions:", Counter([p.suffix.lower() for p in test_imgs]).most_common())


Train total items: 30616
Test total items: 7655
Train images: 30612
Test images: 7653
Train extensions: [('.jpg', 30612)]
Test extensions: [('.jpg', 7653)]


In [66]:
import pandas as pd

df = pd.read_csv(TRAIN_CSV)
print(df.head())
print(df.columns)
print(df.dtypes)
print("rows:", len(df))

      img_name  label
0  train_1.jpg     21
1  train_2.jpg     29
2  train_3.jpg     17
3  train_4.jpg     21
4  train_5.jpg     50
Index(['img_name', 'label'], dtype='str')
img_name      str
label       int64
dtype: object
rows: 30612


In [67]:
# pas deze kolomnamen aan naar wat jij in df.columns ziet
# voorbeeld:
IMG_COL = "img_name"   # of "filename"
LBL_COL = "label"      # of "class_id"

# als in csv geen extensie staat, voeg die later toe (dataset afhankelijk)
missing = []
for x in df[IMG_COL].astype(str).head(200):  # eerst sample (200)
    # probeer direct pad, of met .jpg als nodig
    p1 = TRAIN_DIR / x
    p2 = TRAIN_DIR / (x + ".jpg")
    if not (p1.exists() or p2.exists()):
        missing.append(x)

print("Missing (sample):", len(missing))
print(missing[:10])

Missing (sample): 200
['train_1.jpg', 'train_2.jpg', 'train_3.jpg', 'train_4.jpg', 'train_5.jpg', 'train_6.jpg', 'train_7.jpg', 'train_8.jpg', 'train_9.jpg', 'train_10.jpg']


In [68]:
lines = CLASS_TXT.read_text(encoding="utf-8").splitlines()
print("num classes (lines):", len(lines))
print("first 5:", lines[:5])

mapping = {}
for ln in lines:
    parts = ln.strip().split()
    if len(parts) >= 2 and parts[0].isdigit():
        mapping[int(parts[0])] = " ".join(parts[1:])
print("example mapping:", list(mapping.items())[:5])

num classes (lines): 81
first 5: ['1 lasagna', '2 grilled_cheese_sandwich', '3 strudel', '4 greek_salad', '5 scotch_egg']
example mapping: [(1, 'lasagna'), (2, 'grilled_cheese_sandwich'), (3, 'strudel'), (4, 'greek_salad'), (5, 'scotch_egg')]


In [69]:
from PIL import Image
import random
from collections import Counter

sample = random.sample(train_imgs, k=min(200, len(train_imgs)))

sizes = []
modes = Counter()
bad = 0

for p in sample:
    try:
        with Image.open(p) as im:
            sizes.append(im.size)         # (width, height)
            modes[im.mode] += 1           # "RGB", "L", "RGBA", ...
    except Exception:
        bad += 1

print("Sampled:", len(sample))
print("Bad images in sample:", bad)
print("Modes:", modes)
print("Example sizes:", sizes[:10])

ws = [w for (w,h) in sizes]
hs = [h for (w,h) in sizes]
print("Width min/max:", min(ws), max(ws))
print("Height min/max:", min(hs), max(hs))


Sampled: 200
Bad images in sample: 0
Modes: Counter({'RGB': 200})
Example sizes: [(256, 256), (256, 341), (341, 256), (384, 256), (384, 256), (256, 256), (454, 256), (384, 256), (256, 256), (256, 319)]
Width min/max: 256 557
Height min/max: 256 1408


In [70]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[LBL_COL] if LBL_COL in df.columns else None
)

print(len(train_df), len(val_df))

24489 6123


In [71]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class FoodDataset(Dataset):
    def __init__(self, dataframe, img_dir, img_col, lbl_col=None, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_col = img_col
        self.lbl_col = lbl_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, x: str):
        p = self.img_dir / x
        if p.exists():
            return p
        # fallback als csv ids zonder extensie zijn
        for ext in [".jpg", ".jpeg", ".png"]:
            p2 = self.img_dir / (x + ext)
            if p2.exists():
                return p2
        return p  # zal later error geven

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row[self.img_col])
        path = self._resolve_path(img_id)

        with Image.open(path) as im:
            im = im.convert("RGB")  # force 3 channels
            if self.transform:
                im = self.transform(im)

        if self.lbl_col is None:
            return im, img_id  # test-set (geen label)
        label = int(row[self.lbl_col])
        return im, label

In [72]:
from torchvision import transforms

IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),  # (H,W,C) -> (C,H,W) en /255
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [73]:
from torch.utils.data import DataLoader

train_ds = FoodDataset(train_df, TRAIN_DIR, IMG_COL, LBL_COL, transform=train_tfms)
val_ds   = FoodDataset(val_df,   TRAIN_DIR, IMG_COL, LBL_COL, transform=val_tfms)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# sanity check: 1 batch pakken
x, y = next(iter(train_loader))
print("x shape:", x.shape)  # verwacht: [B, 3, 224, 224]
print("y shape:", y.shape)  # verwacht: [B]
print("y example:", y[:10])

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/dl2025_cpu/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/dl2025_cpu/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'FoodDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


KeyboardInterrupt: 

In [ ]:
num_classes = df[LBL_COL].nunique()
print("num_classes:", num_classes)
print("min/max label:", df[LBL_COL].min(), df[LBL_COL].max())

### Training

In [ ]:
def train_model(
    model,
    train_loader,
    test_loader,
    batch_size,
    epochs,
    seed,
    lr,
    device,
    data_dir="runs/fashionmnist_experiment"
):
    # seed for reproducability
    set_seed(seed)
    # tensorboard
    writer = SummaryWriter(data_dir)
    model_plotted = False

    # loss module and optimizer
    loss_module = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # initializing best tracking
    best_val_acc = -1.0
    best_epoch = -1
    best_model = deepcopy(model)

    for epoch in tqdm(range(epochs)):
        model.train()

        running_loss = 0.0
        total_correct = 0
        total_samples = 0

        for inputs, targets in train_loader:
            # push to device
            inputs = inputs.to(device)
            targets = targets.to(device)

            if not model_plotted:
                writer.add_graph(model, inputs)
                model_plotted = True
            # set gradient to nothing
            optimizer.zero_grad()

            # forward
            logits = model(inputs)
            loss = loss_module(logits, targets)

            # backward + update
            loss.backward()
            optimizer.step()

            # stats
            running_loss += loss.item()

            # predictions is the output of the system
            preds = logits.argmax(dim=1)
            total_correct += (preds == targets).sum().item()
            total_samples += targets.size(0)

        # epoch metrics (outside batch loop!)
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = total_correct / total_samples

        # write to tensorboard
        writer.add_scalar("Loss/train", epoch_loss, epoch)
        writer.add_scalar("Accuracy/train", epoch_acc, epoch)

        # validation
        val_acc = eval_model(model, test_loader)  # uses no_grad + eval inside
        writer.add_scalar("Accuracy/val", val_acc, epoch)

        # checkpoint
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
        }, "checkpoint.pth")

        # track best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_model = deepcopy(model)

    # Test best model
    best_test_acc = eval_model(best_model, test_loader)
    writer.add_scalar("Accuracy/best_test", best_test_acc, epochs)

    writer.close()

    torch.save(best_model.state_dict(), "best_model.pth")

In [ ]:
model = CNN()

train_model(model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            device=device,
            batch_size=32,
            epochs=2,
            seed=42,
            lr=0.1)

NameError: name 'test_loader' is not defined